In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [2]:
%%writefile src/model.py
import numpy as np
from src import metrics

def log5(pa,pb):
    return (pa-pa*pb)/(pa+pb-2*pa*pb)
def game_prob(pa,pb,a_is_home,hfa=0.023):
    p=log5(pa,pb)+(hfa if a_is_home else -hfa)
    return min(max(p,0.05),0.95)


Writing src/model.py


In [4]:
from src import model

print(model.log5(0.5,0.5))
print(model.log5(0.65,0.5))

0.5
0.6500000000000001


In [22]:
%%writefile -a src/model.py


def simulate_series(pa,pb,home_pattern,n=10000,hfa=0.023,seed=2026,need=4):
    rng=np.random.default_rng(seed)
    prob=[game_prob(pa,pb,h,hfa) for h in home_pattern]
    a_title=0
    results={}
    for _ in range(n):
        wa=wb=0
        for p in prob:
            if rng.random()< p:
                wa+=1
            else:
                wb+=1
            if wa==need or wb==need:
                break
        a_title+=1 if wa==need else 0
        key=f'{wa}-{wb}'
        results[key]=results.get(key,0)+1
    return a_title/n, results, prob


Appending to src/model.py


In [4]:
import pandas as pd
from src import model,config

team=pd.read_csv('data/clean/teams.csv').set_index('team')

p , results,prob=model.simulate_series(team.loc['味全龍','pythag'],team.loc['中信兄弟','pythag'],config.TS_HOME_PATTERN)
print(p)
print(results)

0.6763
{'4-1': 1918, '2-4': 1015, '1-4': 737, '4-0': 1095, '4-2': 2075, '4-3': 1675, '3-4': 1132, '0-4': 353}


In [6]:
n=[100,1000,100000]

import pandas as pd
from src import model,config

team=pd.read_csv('data/clean/teams.csv').set_index('team')
for time in n:
    p , _,_=model.simulate_series(team.loc['味全龍','pythag'],team.loc['中信兄弟','pythag'],config.TS_HOME_PATTERN,time,hfa=0)
    print(p)

0.62
0.632
0.65319
